In [1]:
#LSTM

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau



In [2]:
FILE_PATH = "GOOG.csv"
TIME_STEP = 60
TRAIN_RATIO = 0.80
EPOCHS = 50
BATCH_SIZE = 32
SEED = 42

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [3]:
df = pd.read_csv(FILE_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df.set_index("date")

In [4]:
df["ma_5"] = df["close"].rolling(5).mean()
df["ma_20"] = df["close"].rolling(20).mean()
df["return_1"] = df["close"].pct_change()
df["volatility_5"] = df["return_1"].rolling(5).std()
df["target_return"] = df["close"].shift(-1) / df["close"] - 1

df = df.dropna().copy()

In [5]:
feature_columns = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "ma_5",
    "ma_20",
    "return_1",
    "volatility_5"
]

target_column = "target_return"


train_size = int(len(df) * TRAIN_RATIO)

train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Train date range:", train_df.index.min(), "->", train_df.index.max())
print("Test date range :", test_df.index.min(), "->", test_df.index.max())

Train shape: (990, 18)
Test shape : (248, 18)
Train date range: 2016-07-12 00:00:00+00:00 -> 2020-06-16 00:00:00+00:00
Test date range : 2020-06-17 00:00:00+00:00 -> 2021-06-10 00:00:00+00:00


In [6]:
scaler = StandardScaler()

scaled_train_features = scaler.fit_transform(train_df[feature_columns])

test_input_df = df.iloc[train_size - TIME_STEP:].copy()
scaled_test_features = scaler.transform(test_input_df[feature_columns])


def create_sequences(features, target_returns, close_values, dates, time_step):
    X, y = [], []
    prev_close_list = []
    next_close_list = []
    pred_date_list = []

    for end_idx in range(time_step, len(features)):
        X.append(features[end_idx - time_step:end_idx])

        y.append(target_returns[end_idx - 1])

        prev_close_list.append(close_values[end_idx - 1])
        next_close_list.append(close_values[end_idx])
        pred_date_list.append(dates[end_idx])

    return (
        np.array(X, dtype=np.float32),
        np.array(y, dtype=np.float32),
        np.array(prev_close_list, dtype=np.float32),
        np.array(next_close_list, dtype=np.float32),
        np.array(pred_date_list)
    )

In [7]:
X_train, y_train, prev_close_train, next_close_train, pred_dates_train = create_sequences(
    features=scaled_train_features,
    target_returns=train_df[target_column].values,
    close_values=train_df["close"].values,
    dates=train_df.index.values,
    time_step=TIME_STEP
)

X_test, y_test, prev_close_test, next_close_test, pred_dates_test = create_sequences(
    features=scaled_test_features,
    target_returns=test_input_df[target_column].values,
    close_values=test_input_df["close"].values,
    dates=test_input_df.index.values,
    time_step=TIME_STEP
)

print("\nSequence shapes")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)


Sequence shapes
X_train: (930, 60, 9)
y_train: (930,)
X_test : (248, 60, 9)
y_test : (248,)


In [ ]:
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),

    LSTM(96, return_sequences=True),
    Dropout(0.10),

    LSTM(48, return_sequences=False),
    Dropout(0.10),

    Dense(24, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=4,
    min_lr=1e-6
)

history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    shuffle=False,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
y_pred_return = model.predict(X_test, verbose=0).flatten()

pred_close = prev_close_test * (1 + y_pred_return)
actual_close = next_close_test

result_df = pd.DataFrame({
    "Actual_Return": y_test,
    "Predicted_Return": y_pred_return,
    "Actual_Close": actual_close,
    "Predicted_Close": pred_close
}, index=pd.to_datetime(pred_dates_test))

print("\nResult preview:")
print(result_df.head(10))

In [ ]:
mae = mean_absolute_error(result_df["Actual_Close"], result_df["Predicted_Close"])
rmse = np.sqrt(mean_squared_error(result_df["Actual_Close"], result_df["Predicted_Close"]))
r2 = r2_score(result_df["Actual_Close"], result_df["Predicted_Close"])

print("\nTest Metrics")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2  : {r2:.4f}")

plt.figure(figsize=(12, 5))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Training History")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(result_df.index, result_df["Actual_Close"], label="Real Price")
plt.plot(result_df.index, result_df["Predicted_Close"], label="Predicted Price")
plt.title("GOOG - Real vs Predicted Close Price")
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()